# Combinatorics

The counting techniques that decide how big a search space is, how many ways a problem can be arranged, and what answer to print "mod 10⁹+7" — permutations and combinations, Pascal's triangle, `nCr mod p`, Catalan numbers, inclusion–exclusion, and stars-and-bars. The angle, as ever: **know the formula, but reach for the C-level builtin** (`math.comb`, `math.perm`, `math.factorial`) — all *exact*, thanks to arbitrary-precision `int`. Every nontrivial claim below is proven by asserting against `math.comb` or brute-force enumeration over small cases.

This is the *counting* companion to the **number theory** notebook (GCD, sieve, modular inverse) and the **combinatorial generation** pattern (actually *listing* the subsets/permutations rather than counting them).

**Contents**

1. **Counting foundations** — permutations, combinations & Pascal's triangle
2. **nCr mod p** — factorials + Fermat inverses, and **Lucas' theorem**
3. **Catalan numbers** — closed form, DP, and what they count
4. **Inclusion–exclusion** — counting by adding and subtracting overlaps
5. **Stars and bars** — non-negative integer solutions & compositions
6. **When to use + cheat-sheet**


## 1. Counting foundations — permutations, combinations & Pascal's triangle

Two questions sit under almost every counting problem:

- **Permutations** `P(n, k)` = ordered arrangements of `k` items from `n` = `n! / (n−k)!`.
- **Combinations** `C(n, k)` = unordered selections of `k` from `n` = `n! / (k!·(n−k)!)`, written `nCk` or "n choose k".

The standard library gives all three as **exact** C-level builtins — `math.factorial`, `math.perm`, `math.comb` — and because Python ints are arbitrary-precision, `C(52, 5)` or `C(100, 50)` never overflows. Reach for these instead of writing your own factorial ratio (which churns through huge intermediates):

In [1]:
import math

print("factorial(5):", math.factorial(5))   # 120
print("perm(5, 2)  :", math.perm(5, 2))      # 20 = 5 * 4        (ordered)
print("comb(5, 2)  :", math.comb(5, 2))      # 10 = 5 * 4 / 2    (unordered)
print("comb(52, 5) :", math.comb(52, 5))     # poker hands, exact
print("comb(100,50):", math.comb(100, 50))   # 30-digit int, no overflow

# prove the defining ratios on small cases, the long way vs the builtin
def perm_def(n, k): return math.factorial(n) // math.factorial(n - k)
def comb_def(n, k): return math.factorial(n) // (math.factorial(k) * math.factorial(n - k))
assert all(math.perm(n, k) == perm_def(n, k) for n in range(13) for k in range(n + 1))
assert all(math.comb(n, k) == comb_def(n, k) for n in range(13) for k in range(n + 1))
print("perm/comb match the factorial definitions: OK")


factorial(5): 120
perm(5, 2)  : 20
comb(5, 2)  : 10
comb(52, 5) : 2598960
comb(100,50): 100891344545564193334812497256
perm/comb match the factorial definitions: OK


**Pascal's triangle** is the table of all `C(n, k)`, and it's built without a single factorial via the recurrence

$$C(n, k) = C(n-1, k-1) + C(n-1, k),\qquad C(n, 0) = C(n, n) = 1.$$

(Each way to choose `k` from `n` either uses the last element — `C(n−1, k−1)` ways — or doesn't — `C(n−1, k)` ways.) That's the O(n²)-additions trick behind every `nCr` DP table, and it stays in small-int arithmetic the whole way. Let's build the whole triangle from the recurrence and assert every entry equals `math.comb`:

In [2]:
def pascal(rows):
    tri = [[1]]
    for n in range(1, rows):
        prev = tri[-1]
        # interior entries are the sum of the two above; edges are 1
        row = [1] + [prev[k - 1] + prev[k] for k in range(1, n)] + [1]
        tri.append(row)
    return tri

tri = pascal(15)

# PROVE: every entry built from the recurrence equals math.comb(n, k)
assert all(tri[n][k] == math.comb(n, k) for n in range(len(tri)) for k in range(n + 1))
print("Pascal's triangle matches math.comb for all 0 <= k <= n < 15: OK\n")

for n in range(8):                                   # show the top of it
    print(" " * (8 - n), "  ".join(f"{v:2d}" for v in tri[n]))


Pascal's triangle matches math.comb for all 0 <= k <= n < 15: OK

          1
         1   1
        1   2   1
       1   3   3   1
      1   4   6   4   1
     1   5  10  10   5   1
    1   6  15  20  15   6   1
   1   7  21  35  35  21   7   1


## 2. `nCr mod p` — factorials + Fermat inverses, and Lucas' theorem

Counting problems often ask for the answer **modulo a prime** `p` (classically `10⁹+7`) so it fits in a machine word. You can't just compute `C(n, k)` and take `% p` once `n` is large — but you *can* work mod `p` throughout if you can **divide** mod `p`, i.e. multiply by a **modular inverse**.

The recipe — precompute once, then answer each query in O(1):

1. `fact[i] = i! mod p` for `i` up to the max `n`.
2. `inv_fact[i] = (i!)⁻¹ mod p`, the inverse of each factorial.
3. `C(n, k) mod p = fact[n] · inv_fact[k] · inv_fact[n−k] mod p`.

The inverse comes from **Fermat's little theorem**: for prime `p` and `a` not divisible by `p`, `a^(p−1) ≡ 1`, so `a⁻¹ ≡ a^(p−2) (mod p)` — computed by the C-level `pow(a, p-2, p)`. (Python 3.8+ also exposes it directly as `pow(a, -1, p)`. See the **number theory** notebook for the modular-inverse details.) A neat trick keeps it to a *single* `pow`: invert the top factorial, then walk `inv_fact` down via `inv_fact[i-1] = inv_fact[i] · i`.

In [3]:
P = 10 ** 9 + 7
N = 2000                                              # max n we'll query

fact = [1] * (N + 1)
for i in range(1, N + 1):
    fact[i] = fact[i - 1] * i % P

inv_fact = [1] * (N + 1)
inv_fact[N] = pow(fact[N], P - 2, P)                  # Fermat: one modular inverse
for i in range(N, 0, -1):                             # then derive the rest cheaply
    inv_fact[i - 1] = inv_fact[i] * i % P

def ncr_mod(n, k):
    if k < 0 or k > n:
        return 0
    return fact[n] * inv_fact[k] % P * inv_fact[n - k] % P

# PROVE against math.comb(n, k) % P over a spread of small and mid cases
assert all(ncr_mod(n, k) == math.comb(n, k) % P
           for n in range(0, 300) for k in range(0, n + 1, 5))
print("ncr_mod matches math.comb % P for sampled n < 300: OK")
print("C(2000, 1000) mod P =", ncr_mod(2000, 1000))


ncr_mod matches math.comb % P for sampled n < 300: OK
C(2000, 1000) mod P = 72475738


That table approach needs `n < p`. When **`n` itself is huge** but the modulus `p` is a **small prime**, use **Lucas' theorem**: write `n` and `k` in base `p` as digit lists `(n₀, n₁, …)` and `(k₀, k₁, …)`, and then

$$C(n, k) \equiv \prod_i C(n_i, k_i) \pmod{p}.$$

Each digit `C(nᵢ, kᵢ)` has both arguments `< p`, so it's a tiny lookup — and if any `kᵢ > nᵢ` the whole product is 0. This handles `n` with hundreds of digits against a small `p`:

In [4]:
def comb_mod_lucas(n, k, p):
    result = 1
    while n or k:
        ni, ki = n % p, k % p                         # base-p digits, low to high
        if ki > ni:
            return 0                                   # a digit can't be chosen -> 0
        result = result * math.comb(ni, ki) % p        # C(ni, ki) with ni, ki < p
        n //= p
        k //= p
    return result

p = 13
# PROVE vs math.comb for every small (n, k)
assert all(comb_mod_lucas(n, k, p) == math.comb(n, k) % p
           for n in range(60) for k in range(n + 1))
print("Lucas matches math.comb % p for all n, k < 60 (p = 13): OK")

# and it scales to enormous n where a factorial table never could:
n, k = 10 ** 18 + 5, 123456789
print("C(10^18 + 5, 123456789) mod 13 =", comb_mod_lucas(n, k, p))
assert comb_mod_lucas(1000, 357, p) == math.comb(1000, 357) % p
print("cross-check C(1000, 357) mod 13:", comb_mod_lucas(1000, 357, p), "== ", math.comb(1000, 357) % p)


Lucas matches math.comb % p for all n, k < 60 (p = 13): OK
C(10^18 + 5, 123456789) mod 13 = 0
cross-check C(1000, 357) mod 13: 6 ==  6


## 3. Catalan numbers — closed form, DP, and what they count

The **Catalan numbers** `1, 1, 2, 5, 14, 42, 132, …` are the answer to a surprising family of counting problems. Two equivalent definitions:

- **Closed form:** $C_n = \dfrac{1}{n+1}\binom{2n}{n} = \dfrac{(2n)!}{(n+1)!\,n!}$ — one `math.comb` away.
- **Recurrence (DP):** $C_0 = 1,\quad C_{n+1} = \sum_{i=0}^{n} C_i\,C_{n-i}$ — the "split at the first matching pair" structure, O(n²).

They count, among many things: **balanced-parenthesis strings** of `n` pairs, **shapes of binary search trees** with `n` nodes, triangulations of a polygon, and monotonic lattice paths under the diagonal. Let's compute them both ways and prove the count by brute-force enumeration:

In [5]:
from itertools import product

def catalan_closed(n):
    return math.comb(2 * n, n) // (n + 1)

def catalan_dp(n):
    C = [1] + [0] * n
    for m in range(1, n + 1):
        C[m] = sum(C[i] * C[m - 1 - i] for i in range(m))   # split at first pair
    return C[n]

closed = [catalan_closed(n) for n in range(10)]
dp = [catalan_dp(n) for n in range(10)]
print("closed form:", closed)
print("DP recurrence:", dp)
assert closed == dp                                          # two routes agree
assert closed[:7] == [1, 1, 2, 5, 14, 42, 132]
print("closed form == DP recurrence: OK")


closed form: [1, 1, 2, 5, 14, 42, 132, 429, 1430, 4862]
DP recurrence: [1, 1, 2, 5, 14, 42, 132, 429, 1430, 4862]
closed form == DP recurrence: OK


In [6]:
# PROOF 1: C_n counts balanced-parenthesis strings of n pairs
def count_balanced(n):
    total = 0
    for combo in product("()", repeat=2 * n):       # all 2^(2n) bracket strings
        bal = 0
        ok = True
        for c in combo:
            bal += 1 if c == "(" else -1
            if bal < 0:                              # ")" with nothing open
                ok = False
                break
        if ok and bal == 0:
            total += 1
    return total

# PROOF 2: C_n counts the distinct shapes of a BST (binary tree) with n nodes
def count_bst_shapes(n):
    if n <= 1:
        return 1
    return sum(count_bst_shapes(i) * count_bst_shapes(n - 1 - i) for i in range(n))

for n in range(7):
    assert count_balanced(n) == catalan_closed(n)
    assert count_bst_shapes(n) == catalan_closed(n)
print("balanced-paren count  :", [count_balanced(n) for n in range(7)])
print("BST-shape count       :", [count_bst_shapes(n) for n in range(7)])
print("Catalan closed form   :", [catalan_closed(n) for n in range(7)])
print("\nC_n counts both, verified by brute force for n <= 6: OK")


balanced-paren count  : [1, 1, 2, 5, 14, 42, 132]
BST-shape count       : [1, 1, 2, 5, 14, 42, 132]
Catalan closed form   : [1, 1, 2, 5, 14, 42, 132]

C_n counts both, verified by brute force for n <= 6: OK


## 4. Inclusion–exclusion — counting by adding and subtracting overlaps

To count items in a union of overlapping sets, you can't just add the sizes — you'd double-count the overlaps. **Inclusion–exclusion** fixes it by alternating: add singles, subtract pairs, add triples, …

$$\Bigl|\,\bigcup A_i\,\Bigr| = \sum |A_i| - \sum |A_i \cap A_j| + \sum |A_i \cap A_j \cap A_k| - \cdots$$

A classic use: **how many integers in `[1, N]` are divisible by *none* of a set of primes?** The count divisible by a subset `S` of primes is `N // (product of S)` (they're coprime, so a number divisible by all of them is divisible by their product). So we sum over all subsets with sign `(−1)^|S|` — equivalently, this counts the numbers coprime to every listed prime:

In [7]:
from itertools import combinations

def count_coprime(N, primes):
    """Count of x in [1, N] divisible by none of `primes`, via inclusion-exclusion."""
    total = 0
    for r in range(len(primes) + 1):
        for subset in combinations(primes, r):
            prod = 1
            for q in subset:
                prod *= q
            total += (-1) ** r * (N // prod)       # +empty set (N), -singles, +pairs...
    return total

N, primes = 100, [2, 3, 5]
brute = sum(all(x % q for q in primes) for x in range(1, N + 1))
assert count_coprime(N, primes) == brute
print(f"in [1, {N}], divisible by none of {primes}: {count_coprime(N, primes)} (brute: {brute})")

# verify across many N and prime sets
for N in (50, 100, 1000):
    for primes in ([2, 3], [2, 3, 5], [2, 3, 5, 7], [3, 7, 11]):
        brute = sum(all(x % q for q in primes) for x in range(1, N + 1))
        assert count_coprime(N, primes) == brute
print("inclusion-exclusion matches brute force across all tested N and prime sets: OK")


in [1, 100], divisible by none of [2, 3, 5]: 26 (brute: 26)
inclusion-exclusion matches brute force across all tested N and prime sets: OK


The same alternating-sign machinery counts **derangements** `D_n` — permutations with *no* fixed point (nobody gets their own hat back). Inclusion–exclusion over "position `i` is fixed" gives

$$D_n = n!\sum_{i=0}^{n}\frac{(-1)^i}{i!},\qquad\text{equivalently}\quad D_n = \operatorname{round}\!\bigl(n!/e\bigr)\ (n\ge 1).$$

Both forms are easy to verify against brute-force enumeration of permutations:

In [8]:
from itertools import permutations

def derangements_ie(n):
    # inclusion-exclusion: n! * sum_{i=0}^n (-1)^i / i!
    return sum((-1) ** i * math.factorial(n) // math.factorial(i) for i in range(n + 1))

def derangements_brute(n):
    return sum(all(p[i] != i for i in range(n)) for p in permutations(range(n)))

ie = [derangements_ie(n) for n in range(8)]
brute = [derangements_brute(n) for n in range(8)]
nearest = [1] + [round(math.factorial(n) / math.e) for n in range(1, 8)]
print("inclusion-exclusion:", ie)
print("brute force        :", brute)
print("round(n!/e)        :", nearest)
assert ie == brute == nearest
print("\nD_n: all three agree for n <= 7: OK   (D_4 = 9, the 9 derangements of 4 items)")


inclusion-exclusion: [1, 0, 1, 2, 9, 44, 265, 1854]
brute force        : [1, 0, 1, 2, 9, 44, 265, 1854]
round(n!/e)        : [1, 0, 1, 2, 9, 44, 265, 1854]

D_n: all three agree for n <= 7: OK   (D_4 = 9, the 9 derangements of 4 items)


## 5. Stars and bars — non-negative integer solutions & compositions

How many ways can you write `n` as an ordered sum of `k` **non-negative** integers — i.e. solutions to `x₁ + x₂ + ⋯ + x_k = n` with each `xᵢ ≥ 0`? Picture `n` identical stars in a row and `k−1` bars dropped among them to split the stars into `k` groups; any arrangement of `n` stars and `k−1` bars is a solution. There are `n + k − 1` symbols, choose which `k − 1` are bars:

$$\#\{x_i \ge 0 : \textstyle\sum x_i = n\} = \binom{n + k - 1}{k - 1}.$$

For **strictly positive** parts (`xᵢ ≥ 1`, called **compositions** of `n` into `k` parts), give each part one star up front, leaving `n − k` to distribute: `C(n − 1, k − 1)`. Both are one `math.comb`:

In [9]:
from itertools import product

def solutions_nonneg(n, k):
    return math.comb(n + k - 1, k - 1)          # x_i >= 0

def compositions(n, k):
    return math.comb(n - 1, k - 1)              # x_i >= 1

# PROVE non-negative count by enumerating all k-tuples that sum to n
def brute_nonneg(n, k):
    return sum(sum(t) == n for t in product(range(n + 1), repeat=k))

for n in range(0, 8):
    for k in range(1, 5):
        assert solutions_nonneg(n, k) == brute_nonneg(n, k)
print("stars-and-bars (x_i >= 0) matches brute force for n < 8, k <= 4: OK")

# PROVE positive-part compositions by enumeration
def brute_comp(n, k):
    return sum(sum(t) == n for t in product(range(1, n + 1), repeat=k))

for n in range(1, 9):
    for k in range(1, n + 1):
        assert compositions(n, k) == brute_comp(n, k)
print("compositions   (x_i >= 1) matches brute force for n < 9: OK")

print("\nways to make 5 from 3 non-negatives:", solutions_nonneg(5, 3),
      "  e.g.", [t for t in product(range(6), repeat=3) if sum(t) == 5][:3], "...")


stars-and-bars (x_i >= 0) matches brute force for n < 8, k <= 4: OK


compositions   (x_i >= 1) matches brute force for n < 9: OK

ways to make 5 from 3 non-negatives: 21   e.g. [(0, 0, 5), (0, 1, 4), (0, 2, 3)] ...


A useful corollary: summing `compositions(n, k)` over all `k` from 1 to `n` counts the **total compositions** of `n` — every way to write `n` as an ordered sum of positive parts — which famously equals `2^(n−1)`. One more brute-checkable identity:

In [10]:
def total_compositions(n):
    return sum(math.comb(n - 1, k - 1) for k in range(1, n + 1))

def brute_total_compositions(n):
    # count ordered positive sums equal to n, over every part-count k
    total = 0
    for k in range(1, n + 1):
        total += sum(sum(t) == n for t in product(range(1, n + 1), repeat=k))
    return total

for n in range(1, 9):
    assert total_compositions(n) == 2 ** (n - 1) == brute_total_compositions(n)
print("total compositions of n == 2^(n-1), verified vs brute force for n < 9: OK")
print("counts:", [total_compositions(n) for n in range(1, 9)])


total compositions of n == 2^(n-1), verified vs brute force for n < 9: OK
counts: [1, 2, 4, 8, 16, 32, 64, 128]


## 6. When to use + cheat-sheet

| You're asked... | Reach for | Note |
|---|---|---|
| ordered arrangements of `k` from `n` | `math.perm(n, k)` | `= n!/(n−k)!` |
| unordered selections of `k` from `n` | `math.comb(n, k)` | exact, never overflows |
| a whole table of `nCk` | Pascal recurrence | O(n²) adds, small ints |
| `nCk` mod prime `p`, `n < p` | factorials + Fermat inverse | O(1)/query after O(n) setup |
| `nCk` mod *small* prime, `n` huge | **Lucas' theorem** | per base-`p` digit |
| count balanced parens / BST shapes / triangulations | **Catalan** `Cₙ` | `comb(2n,n)/(n+1)` |
| size of a union of overlapping sets / "divisible by none" / derangements | **inclusion–exclusion** | alternate +/− over subsets |
| `x₁+⋯+x_k = n`, `xᵢ ≥ 0` | **stars & bars** `C(n+k−1, k−1)` | `≥ 1`: `C(n−1, k−1)` |

When you actually need to *list* the arrangements (not just count them), that's the **combinatorial generation** pattern (`itertools.product` / `combinations` / `permutations` and backtracking). For the modular-arithmetic foundations — `pow(a, -1, m)`, Fermat, the sieve — see the **number theory** notebook. Arbitrary-precision `int` (the **bit-manipulation** notebook) is what makes every formula here *exact*.

| Quantity | Formula | Complexity | Builtin |
|---|---|:---:|---|
| `n!` | — | O(n) | `math.factorial(n)` |
| `P(n, k)` | `n!/(n−k)!` | O(n) | `math.perm(n, k)` |
| `C(n, k)` | `n!/(k!(n−k)!)` | O(min(k, n−k)) | `math.comb(n, k)` |
| Pascal's triangle | `C(n,k)=C(n−1,k−1)+C(n−1,k)` | O(n²) | — |
| `nCk mod p` (`n<p`) | factorials + inverse | O(n) prep, O(1) query | `pow(a,-1,p)` |
| `nCk mod p` (large `n`) | Lucas: `∏ C(nᵢ,kᵢ)` | O(log_p n) | — |
| Catalan `Cₙ` | `C(2n,n)/(n+1)` | O(n) | `math.comb` |
| inclusion–exclusion | `Σ (−1)^\|S\| \|∩S\|` | O(2^m) subsets | — |
| stars & bars (`≥0`) | `C(n+k−1, k−1)` | O(k) | `math.comb` |
| compositions (`≥1`) | `C(n−1, k−1)` | O(k) | `math.comb` |
